In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RetailX_RDD_Processing") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
rdd = sc.textFile("sales_data.csv")

In [ ]:
header = rdd.first()
rdd = rdd.filter(lambda row: row != header)

In [ ]:
def parse_line(line):
    fields = line.split(",")
    return (
        fields[2],              # product_id
        int(fields[4]),         # quantity
        float(fields[5])        # price
    )

mapped_rdd = rdd.map(parse_line)

In [ ]:
filtered_rdd = mapped_rdd.filter(
    lambda x: x[1] > 0 and x[2] > 0
)

In [ ]:
sales_rdd = filtered_rdd.map(
    lambda x: (x[0], x[1] * x[2])   # (product_id, total_price)
)

In [ ]:
result_rdd = sales_rdd.reduceByKey(
    lambda a, b: a + b
)

In [ ]:
results = result_rdd.collect()

for r in results:
    print(r)

('201', 3000.0)
('202', 2100.0)
('210', 1500.0)
('203', 1200.0)
('204', 2800.0)
('205', 2700.0)
('206', 1800.0)
('207', 3200.0)
('208', 1000.0)
('209', 6000.0)


In [ ]:
top_products = result_rdd.sortBy(lambda x: x[1], ascending=False)

print(top_products.take(5))

[('209', 6000.0), ('207', 3200.0), ('201', 3000.0), ('204', 2800.0), ('205', 2700.0)]


In [ ]:
# Smart Retail Analytics - Phase 2 (DataFrame + Spark SQL)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, month

# -----------------------------------
# 1. Initialize Spark Session
# -----------------------------------
spark = SparkSession.builder \
    .appName("RetailX_DataFrame_SQL") \
    .getOrCreate()

print("🚀 Phase 2 Started - DataFrames & Spark SQL")

# -----------------------------------
# 2. Load Data (CSV)
# -----------------------------------
sales_df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)
customer_df = spark.read.csv("customer_data.csv", header=True, inferSchema=True)
product_df = spark.read.csv("product_data.csv", header=True, inferSchema=True)

# -----------------------------------
# 3. Convert RDD → DataFrame (if needed)
# -----------------------------------
# (Example: if you had RDD from Phase 1)
# sales_df = rdd.toDF(["product_id", "quantity", "price"])

# -----------------------------------
# 4. Filtering High-Value Transactions
# -----------------------------------
high_value_df = sales_df.filter(col("price") > 800)

print("\n💰 High Value Transactions:")
high_value_df.show()

# -----------------------------------
# 5. Join Datasets
# -----------------------------------
joined_df = sales_df \
    .join(customer_df, "customer_id") \
    .join(product_df, "product_id")

print("\n🔗 Joined Data:")
joined_df.show()

# -----------------------------------
# 6. Aggregation → Total Revenue per City
# -----------------------------------
revenue_city_df = joined_df.groupBy("city") \
    .agg(sum(col("quantity") * col("price")).alias("total_revenue"))

print("\n🏙️ Revenue per City:")
revenue_city_df.show()

# -----------------------------------
# 7. Register Temporary Views (for SQL)
# -----------------------------------
joined_df.createOrReplaceTempView("sales_data")

# -----------------------------------
# 8. Spark SQL → Top 5 Products by Revenue
# -----------------------------------
top_products = spark.sql("""
    SELECT product_id,
           SUM(quantity * price) AS revenue
    FROM sales_data
    GROUP BY product_id
    ORDER BY revenue DESC
    LIMIT 5
""")

print("\n🏆 Top 5 Products:")
top_products.show()

# -----------------------------------
# 9. Spark SQL → Monthly Sales Trend
# -----------------------------------
# First convert timestamp to proper format
from pyspark.sql.functions import to_timestamp

joined_df = joined_df.withColumn(
    "timestamp", to_timestamp("timestamp")
)

joined_df.createOrReplaceTempView("sales_data2")

monthly_trend = spark.sql("""
    SELECT MONTH(timestamp) AS month,
           SUM(quantity * price) AS total_sales
    FROM sales_data2
    GROUP BY MONTH(timestamp)
    ORDER BY month
""")

print("\n📈 Monthly Sales Trend:")
monthly_trend.show()

# -----------------------------------
# 10. Stop Spark
# -----------------------------------
spark.stop()

print("\n✅ Phase 2 Completed Successfully!")

🚀 Phase 2 Started - DataFrames & Spark SQL

💰 High Value Transactions:
+--------------+-----------+----------+--------+--------+-----+----------+
|transaction_id|customer_id|product_id|store_id|quantity|price| timestamp|
+--------------+-----------+----------+--------+--------+-----+----------+
|             5|        105|       205|       3|       1|  900|2026-01-05|
|            10|        105|       205|       3|       2|  900|2026-01-10|
|            14|        109|       209|       1|       4| 1000|2026-01-14|
|            19|        109|       209|       1|       2| 1000|2026-01-19|
+--------------+-----------+----------+--------+--------+-----+----------+


🔗 Joined Data:
+----------+-----------+--------------+--------+--------+-----+----------+-----+---------+-------+-----------+-------+
|product_id|customer_id|transaction_id|store_id|quantity|price| timestamp| name|     city|segment|   category|  brand|
+----------+-----------+--------------+--------+--------+-----+----------+